# 02 · Ablation and calibration

**UZ:** Qaysi metrika nimani qo'shadi: bitta metrikali, to'liq va «bittasiz» modellar (NNLS, leave-one-out). Kalibrlangan modelni topshiriqqa qo'llash.

**EN:** Leave-one-out ablation of the metric set and expert calibration of the scoring model.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from cgaaf.course import Course
# Kurs papkasi / course folder (change to your real course):
COURSE = Path("../demo_course")
if not COURSE.exists():          # synthetic sample, for testing only
    from cgaaf.demo import make_demo
    make_demo(COURSE, n_students=24, log=lambda s: None)
c = Course(COURSE)
print(c.meta)

In [ ]:
from cgaaf.reports import ablation, rated
from cgaaf.scoring import keys_for
df = c.dataset()
out = {}
for t in c.tasks():
    d = rated(df[df.task_id == t['id']])
    if len(d) >= 10:
        out[t['id']] = ablation(d, keys_for(t['type'])).round(3)
        display(out[t['id']])

In [ ]:
from cgaaf.scoring import calibrate
APPLY = False   # True -> save calibrated model into task.json
for t in c.tasks():
    d = rated(df[df.task_id == t['id']])
    if len(d) < 8: continue
    M = [{k[2:]: r[k] for k in d.columns if k.startswith('m_') and r[k] == r[k]} for _, r in d.iterrows()]
    res = calibrate(M, d.expert_mean.values, keys_for(t['type']))
    print(t['id'], {k: round(v, 2) for k, v in res['b'].items()}, 'b0 =', round(res['b0'], 2), '| LOOCV r =', round(res['r_loo'], 3), 'MAE =', round(res['mae_loo'], 2))
    if APPLY:
        t['calibration'] = {k: res[k] for k in ('b0', 'b', 'n', 'r_loo', 'mae_loo')}; c.save_task(t)

## Screenshot vs Blender re-render
If the course contains both kinds of submissions, compare agreement by `image_source`.

In [ ]:
d = rated(df)
if 'image_source' in d and d.image_source.nunique() > 1:
    from cgaaf.stats import agreement
    for s, g in d.groupby('image_source'):
        if len(g) >= 4: print(s, {k: round(v, 3) for k, v in agreement(g.auto, g.expert_mean, boot=False).items() if k in ('n', 'pearson_r', 'icc21', 'mae')})
else:
    print('only one image source in this course')